In [ ]:
import duckdb
import pandas as pd
import os

DATA_DIR = os.path.expanduser("~/kkbox-churn/data/")
con = duckdb.connect()

def explore_table(name, filepath):
    print(f"\n{'='*50}")
    print(f"=== {name} ===")
    df = con.execute(f"SELECT * FROM read_csv_auto('{filepath}') LIMIT 5").df()
    print(df.dtypes)
    print(df.head())
    count = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{filepath}')").fetchone()[0]
    print(f"Total rows: {count:,}")
    return df

train_path    = DATA_DIR + "train_v2.csv"
members_path  = DATA_DIR + "members_v3.csv"
txn_v2_path   = DATA_DIR + "transactions_v2.csv"
logs_v2_path  = DATA_DIR + "user_logs_v2.csv"
logs_path     = DATA_DIR + "user_logs.csv"

explore_table("TRAIN_V2 (Labels)", train_path)

churn_rate = con.execute(f"SELECT AVG(is_churn) FROM read_csv_auto('{train_path}')").fetchone()[0]
print(f"Churn rate: {churn_rate:.4f}")

explore_table("MEMBERS_V3 (Demographics)", members_path)
explore_table("TRANSACTIONS_V2 (Payments)", txn_v2_path)
explore_table("USER_LOGS_V2 (Behavior)", logs_v2_path)

print(f"\n{'='*50}")
print("=== USER_LOGS FULL (Schema only) ===")
logs_sample = con.execute(f"SELECT * FROM read_csv_auto('{logs_path}') LIMIT 3").df()
print(logs_sample.dtypes)
print(logs_sample.head(3))
count = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{logs_path}')").fetchone()[0]
print(f"Total rows: {count:,}")

con.close()


=== TRAIN_V2 (Labels) ===
msno          str
is_churn    int64
dtype: object
                                           msno  is_churn
0  ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=         1
1  f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=         1
2  zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=         1
3  8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=         1
4  K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=         1
Total rows: 970,960
Churn rate: 0.0899

=== MEMBERS_V3 (Demographics) ===
msno                        str
city                      int64
bd                        int64
gender                      str
registered_via            int64
registration_init_time    int64
dtype: object
                                           msno  city  bd  gender  \
0  Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=     1   0     NaN   
1  +tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=     1   0     NaN   
2  cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=     1   0     NaN   
3  9bzDeJP6sQodK73K

In [ ]:
results=con.execute("""
SELECT * 
FROM read_csv_auto('../data/members_v3.csv')
WHERE gender='female'
ORDER by city ASC
LIMIT 10
""").df()

print(results)

                                           msno  city  bd  gender  \
0  zKk7+2K/EgzGbWpKYUQjMdSVaT7gfpGzccYS4Rx9svk=     1  45  female   
1  1GMu9VHXpkESXD0zX3rJrDpzq0gOIKPNLHHIgVlJH4c=     1   0  female   
2  uaJm6jEM3/wP6PS4UqDKn+RUDo0/oPKsMLzyt9Edvyk=     1  32  female   
3  hpl/Ea6QjYNFkdjQplH+n1F1o+Y66xGCHV7tofDg0yc=     1  28  female   
4  5+taFIC7uisYOGrXvHLXU76Xi3iqGXCCJrS9u9TKkR4=     1  24  female   
5  6w1ye1AADiKzNPEzgJCKz2a7eHIqvOfX80R5jmKWahI=     1  40  female   
6  U8AUiD7XtFhBIl7kh+LelzANqetg+MFEEoo0ZTxGhZI=     1  38  female   
7  VJBXt3dhQviplsub4V3+RAbL8FcEzt1J+eVnSQsBVPg=     1   0  female   
8  wnUePyABGJPwwP9sLo4SHeiJEnupe8Gynixgu/maDp8=     1  23  female   
9  IColgNiZS288fs/NYq/Avak6CvUefQdfuhFNjEhh7d8=     1   0  female   

   registered_via  registration_init_time  
0               3                20160828  
1               3                20130924  
2               9                20071110  
3               9                20070714  
4               9   

In [ ]:
results=con.execute("""
SELECT city,
       city+ 1,
       gender,
       bd
FROM read_csv_auto('../data/members_v3.csv')
WHERE gender='female'
ORDER by city ASC
LIMIT 10
""").df()

print(results)

   city  (city + 1)  gender  bd
0     1           2  female  32
1     1           2  female  37
2     1           2  female   0
3     1           2  female  23
4     1           2  female  17
5     1           2  female  26
6     1           2  female  53
7     1           2  female  44
8     1           2  female   0
9     1           2  female   0


In [3]:
import duckdb
con=duckdb.connect()

results=con.execute("""
SELECT city,
       gender,
       bd,
       city+ 1 AS 'city example'
FROM read_csv_auto('../data/members_v3.csv')
WHERE gender='female'
ORDER by city ASC
LIMIT 10
""").df()

print(results)

   city  gender  bd  city example
0     1  female  20             2
1     1  female   0             2
2     1  female  41             2
3     1  female  56             2
4     1  female   0             2
5     1  female  38             2
6     1  female   0             2
7     1  female  45             2
8     1  female  43             2
9     1  female  43             2


In [ ]:
results=con.execute("""
    SELECT *
    FROM read_csv_auto("../data/members_v3.csv")
    WHERE NOT (city>=2 AND city<=5)
    ORDER by city ASC
    """).df()

print(results)

                                                 msno  city  bd  gender  \
0        s50p5YEsEoLycWsZdf4CieUn8ltoIPxZpq3j99Sy2U0=     1   0     NaN   
1        pLbWYfSpCIO7Jp/be9fetz3MEhBzY5Fyy3oyHDv9EQ0=     1   0     NaN   
2        Be2Q3/+JxTggPog+lTYK/qManHfqieStE2DUJPltivQ=     1   0     NaN   
3        PyFIpMUo7JLhDdkUPr/SLmiV8fr1TCvAWnOvVkFEHSY=     1   0     NaN   
4        BvTzfcqgguIsEIX3q/ylfE/DOPnf9zDoFG6osOTLH8c=     1   0     NaN   
...                                               ...   ...  ..     ...   
6110269  mazbHanZMnBozFR6kGXQZR2VyPZ0hjzFPs8HEFV7uGw=    22  18    male   
6110270  JChMdrzzc1WCP1LyIWB9b0BVX/ADhU7XeAV7Dyr1hXI=    22  32  female   
6110271  PPbzl5pVwmnOZB2zqPXMxkDJ1t0X7al+vMmbZsPz9B0=    22  31  female   
6110272  MjfymVevGA2qkZ2mm1A2CSg81UAgV9fO2tEago9AbkQ=    22  41  female   
6110273  oKJcorkE7gzTB1Yty1rnCftLJt49hqOF8kC11WdSRkQ=    22  21    male   

         registered_via  registration_init_time  
0                     9                20150720  

In [ ]:
results=con.execute("""
    SELECT *
    FROM read_csv_auto("../data/members_v3.csv")
    WHERE NOT (city>=2 AND city<=5)
    ORDER by city ASC
    """).df()

print(results['city'].unique())

[ 1  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22]


In [ ]:
results=con.execute("""
    SELECT sample.msno, is_churn, bd, gender
    FROM read_csv_auto("../data/sample_submission_v2.csv") sample
    JOIN read_csv_auto("../data/members_v3.csv") members
    ON sample.msno = members.msno
    """).df()

print(results)

                                                msno  is_churn  bd  gender
0       +tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=         0   0     NaN
1       WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=         0  32  female
2       I0yFvqMoNkM8ZNHb617e1RBzIS/YRKemHO7Wj13EtA0=         0  63    male
3       OoDwiKZM+ZGr9P3fRivavgOtglTEaNfWJO4KaJcTTts=         0   0     NaN
4       4De1jAxNRABoyRBDZ82U0yEmzYkqeOugRGVNIf92Xb8=         0  28  female
...                                              ...       ...  ..     ...
795085  mKsBIrIGLIJ2+KWulDMqKVIfK9f2j5leAgOPNRYbKrI=         0   0     NaN
795086  zZmRZFE8NNtQsU6/ZFkn3F7mUYhEcluLVnvEuV8F+v0=         0   0     NaN
795087  D5PEJY8YkLWzvdOT3glpIbXkM8Qh6Cus/dG2Mw7G5Fs=         0   0     NaN
795088  gnED0GiuSfva4AjG5cwOY6OH0IGhWAa2t6VQW7Jth2A=         0   0     NaN
795089  P6poEgvHzZpwWiPnE9DHtPOIJVg8r4dTBfPSHjyalhg=         0   0     NaN

[795090 rows x 4 columns]
